# DiffusionGemma — Multi-Pair Translation
Zero-shot translation using DiffusionGemma W4A16 (GoedelMachines/diffusiongemma-26B-A4B-w4a16), evaluated on all 5 pairs used across this project: EN→RU (WMT14), ZH→EN / EN→ZH (WMT17), JA→EN / EN→JA (WMT20).

Works on **Kaggle or Colab** — Cell 2 auto-detects the platform for secrets.

**Before running:**
- Accelerator: **single GPU only**. This repo's custom `trust_remote_code` loader does not support `device_map="auto"` multi-GPU dispatch — confirmed by `SafetensorError: device auto is invalid` when tried. Use `device_map="cuda"` (single device), same as the project's tested `scripts/infer_diffgemma_w4a16.py`. This means a second T4 on Kaggle won't help — whatever fits on ONE GPU's VRAM is what you get (~16 GB on T4, more on Colab Pro's A100/L4).
- Model card lists ~18-20 GB peak VRAM, which conflicts with this project's own script comment of ~11.5 GB — unverified until an actual run completes past loading. A T4 (16 GB, or ~15 GB usable) may or may not fit; the only way to know is to try Cell 4.
- Internet: ON (Kaggle: toggle in Settings, may need a session restart + phone verification to take effect; Colab: on by default).
- Secrets: add `HF_TOKEN` (from huggingface.co → Settings → Access Tokens) via Kaggle Secrets or Colab Secrets (key icon, left sidebar). You must accept the Gemma Terms of Use on **both** `google/diffusiongemma-26B-A4B-it` and `GoedelMachines/diffusiongemma-26B-A4B-w4a16` before this token can download weights.
- **Always run the smoke test (Cell 5) before the full run (Cell 6).**
- **Session budget**: Kaggle GPU sessions cap at ~9-12h with ~30 GPU-hrs/week on free tier; Colab free tier is similar with less predictable availability. Per-line latency for this model is unbenchmarked. Cell 5's timed smoke test tells you roughly how long all 5 pairs × 3003 lines would take — check it before committing to Cell 6.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q transformers accelerate sacrebleu tqdm

In [ ]:
# Cell 2 — Auth + GPU check (works on Kaggle or Colab)
import os, torch

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("Loaded HF_TOKEN from Kaggle Secrets")
except ModuleNotFoundError:
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        print("Loaded HF_TOKEN from Colab Secrets")
    except (ModuleNotFoundError, Exception):
        from getpass import getpass
        os.environ["HF_TOKEN"] = getpass("Enter HF_TOKEN: ")

print(f"CUDA available: {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}  {p.total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 3 — Download all 5 test sets via sacrebleu (same logic as scripts/vastai_setup.sh)
# Same sentences used in SeqDiffuSeq sessions — BLEU is directly comparable
import sacrebleu, shutil
from pathlib import Path

# pair -> (wmt_set, src_lang_name, tgt_lang_name)
PAIRS = {
    "en-ru": ("wmt14", "English", "Russian"),
    "zh-en": ("wmt17", "Chinese", "English"),
    "en-zh": ("wmt17", "English", "Chinese"),   # flipped zh-en
    "ja-en": ("wmt20", "Japanese", "English"),
    "en-ja": ("wmt20", "English", "Japanese"),  # flipped ja-en
}

def save_pair(wmt, pair, src_ext, tgt_ext, out_dir):
    d = Path(out_dir)
    d.mkdir(parents=True, exist_ok=True)
    src = sacrebleu.get_source_file(wmt, pair)
    ref = sacrebleu.get_reference_files(wmt, pair)[0]
    shutil.copy(src, d / f"test.{src_ext}")
    shutil.copy(ref, d / f"test.{tgt_ext}")

data_root = Path("/kaggle/working/data")

save_pair("wmt14", "en-ru", "en", "ru", data_root / "en-ru")
save_pair("wmt17", "zh-en", "zh", "en", data_root / "zh-en")
# en-zh: flip of zh-en (English becomes source, Chinese becomes reference)
(data_root / "en-zh").mkdir(parents=True, exist_ok=True)
shutil.copy(data_root / "zh-en/test.en", data_root / "en-zh/test.en")
shutil.copy(data_root / "zh-en/test.zh", data_root / "en-zh/test.zh")
save_pair("wmt20", "ja-en", "ja", "en", data_root / "ja-en")
# en-ja: flip of ja-en
(data_root / "en-ja").mkdir(parents=True, exist_ok=True)
shutil.copy(data_root / "ja-en/test.en", data_root / "en-ja/test.en")
shutil.copy(data_root / "ja-en/test.ja", data_root / "en-ja/test.ja")

# Load all pairs into memory: pair -> (src_lines, ref_lines, src_lang, tgt_lang)
lines = {}
for pair, (wmt, src_lang, tgt_lang) in PAIRS.items():
    src_ext, tgt_ext = pair.split("-")
    src = (data_root / pair / f"test.{src_ext}").read_text().splitlines()
    ref = (data_root / pair / f"test.{tgt_ext}").read_text().splitlines()
    lines[pair] = (src, ref, src_lang, tgt_lang)
    print(f"{pair} ({wmt}): {len(src)} lines — {src_lang} → {tgt_lang}")

In [ ]:
# Cell 4 — Load DiffusionGemma W4A16 (single GPU only — this repo's custom
# trust_remote_code loader does NOT support device_map="auto": it forwards the
# literal string to safetensors.safe_open(device=...), which errors on "auto".
# Matches the tested config in scripts/infer_diffgemma_w4a16.py.
from transformers import AutoTokenizer, AutoModelForCausalLM

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

MODEL_ID = "GoedelMachines/diffusiongemma-26B-A4B-w4a16"
HF_TOKEN = os.environ["HF_TOKEN"]

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN, trust_remote_code=True)

print("Loading model (W4A16)... this will take a few minutes on first run")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    trust_remote_code=True,
    device_map="cuda",
)
model.eval()
print(f"GPU memory after loading: {torch.cuda.memory_allocated(0)/1e9:.1f} GB")

In [ ]:
# Cell 5 — Translation helper + timed smoke test (3 lines per pair)
# Run this BEFORE Cell 6 — confirms the model works AND gives a per-line time estimate
# across all 5 pairs before committing to the full ~15,000-line run.
import time

def translate(text, src_lang="English", tgt_lang="Russian", max_new_tokens=150):
    prompt = (
        f"Translate the following {src_lang} sentence to {tgt_lang}. "
        f"Output only the {tgt_lang} translation, nothing else.\n{text}"
    )
    msgs = [{"role": "user", "content": prompt}]
    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
        input_ids = tokenizer.apply_chat_template(
            msgs, add_generation_prompt=True, return_tensors="pt"
        ).to("cuda")
    else:
        input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")
    with torch.no_grad():
        out = model.generate(input_ids, max_new_tokens=max_new_tokens)
    seqs = out.sequences if hasattr(out, "sequences") else out
    full = tokenizer.decode(seqs[0], skip_special_tokens=True)
    for marker in ("model\nthought\n", "model\n"):
        if marker in full:
            full = full.split(marker, 1)[1]
            break
    return full.strip()

print("=== Timed smoke test (3 lines per pair) ===")
per_line_times = []
for pair, (src_lines, ref_lines, src_lang, tgt_lang) in lines.items():
    print(f"\n--- {pair} ({src_lang} → {tgt_lang}) ---")
    for src in src_lines[:3]:
        t0 = time.time()
        hyp = translate(src, src_lang, tgt_lang)
        dt = time.time() - t0
        per_line_times.append(dt)
        print(f"SRC: {src}")
        print(f"HYP: {hyp}")
        print(f"({dt:.1f}s)\n")

avg = sum(per_line_times) / len(per_line_times)
total_lines = sum(len(v[0]) for v in lines.values())
print(f"Avg {avg:.1f}s/line → est. {avg*total_lines/3600:.1f}h for all {total_lines} lines across 5 pairs")

In [ ]:
# Cell 6 — Full evaluation, all 5 pairs (run only after smoke test confirms good output + acceptable time estimate)
from tqdm.notebook import tqdm

results_root = Path("/kaggle/working/results")
summary = []

for pair, (src_lines, ref_lines, src_lang, tgt_lang) in lines.items():
    print(f"\n=== {pair} ({src_lang} → {tgt_lang}): {len(src_lines)} lines ===")
    hyps = [
        translate(line, src_lang, tgt_lang)
        for line in tqdm(src_lines, desc=pair)
    ]

    out_dir = results_root / pair
    out_dir.mkdir(parents=True, exist_ok=True)
    tgt_ext = pair.split("-")[1]
    (out_dir / f"hyps.{tgt_ext}").write_text("\n".join(hyps))

    bleu = sacrebleu.corpus_bleu(hyps, [ref_lines], tokenize="13a")
    report = (
        f"SacreBLEU (13a): {bleu.score:.2f}\n"
        f"Model: GoedelMachines/diffusiongemma-26B-A4B-w4a16\n"
        f"Lines: {len(hyps)}\n"
        f"pair: {pair}\n"
    )
    (out_dir / "bleu_report.txt").write_text(report)
    print(report)
    summary.append((pair, bleu.score, len(hyps)))

print("\n=== Summary ===")
for pair, score, n in summary:
    print(f"{pair}: BLEU {score:.2f} ({n} lines)")